In [1]:
from process import ProcessRNRDataset
threadId, postId, threadTag, structures, posts = ProcessRNRDataset()

charliehebdo rumor: 100%|██████████| 459/459 [00:00<00:00, 775.00it/s]
ebola-essien non-rumor: 0it [00:00, ?it/s]
sydneysiege rumor: 100%|██████████| 523/523 [00:00<00:00, 750.97it/s]


In [3]:
with open('./all/thread_id.txt', 'w') as f:
    for id in threadId:
        f.write(id + '\n')


In [4]:
with open('./all/post_id.txt', 'w') as f:
    for id in postId:
        f.write(id + '\n')
        

In [5]:
with open('./all/thread_label.txt', 'w') as f:
    for id in threadId:
        f.write(str(threadTag[id]) + '\n')

In [6]:
import json
with open('./all/structures.json', 'w') as f:
    json.dump(structures, f)

In [7]:
import json
with open('./all/posts.json', 'w') as f:
    json.dump(posts, f)

In [ ]:
import json
from process import label2Index
with open('./rumorCategory.json', 'w') as f:
    json.dump(label2Index, f)

In [2]:
print(len(postId))
print(len(posts))

104582
104582


In [5]:
for id in postId:
    try:
        a = posts[id]
    except Exception as e:
        print(e)

In [ ]:
import json
with open('./all/posts.json', 'w') as f:
    json.dump(posts, f)

### split dataset into train-dev-test

In [15]:
import os
import json

THREAD_ID_FILE = 'thread_id.txt'
POST_ID_FILE = 'post_id.txt'
THREAD_LABEL_FILE = 'thread_label.txt'
POST_LABEL_FILE = 'post_label.txt'
STRUCTURE_FILE = 'structures.json'
POST_FILE = 'posts.json'
RUMOR_CATEGPRY = 'rumorCategory.json'
STANCE_CATEGPRY = 'stanceCategory.json'

data_path = '.'
type = 'all'
with open(os.path.join(data_path, type, THREAD_ID_FILE)) as f:
    all_thread_id = [line.strip() for line in f.readlines()] # list
with open(os.path.join(data_path, type, POST_ID_FILE)) as f:
    all_post_id = [line.strip() for line in f.readlines()] # list
with open(os.path.join(data_path, type, THREAD_LABEL_FILE)) as f:
    all_thread_label = [line.strip() for line in f.readlines()] # list, 读入的是str形式
with open(os.path.join(data_path, type, STRUCTURE_FILE)) as f:
    all_structure = json.load(f) # dict
with open(os.path.join(data_path, type, POST_FILE)) as f:
    all_post = json.load(f) # dict
with open(os.path.join(data_path, RUMOR_CATEGPRY)) as f:
    category = json.load(f) #dict

In [21]:
print(len(all_post_id))

104582


In [16]:
id2label = {}
for thread_id, thread_label in zip(all_thread_id, all_thread_label):
    id2label[thread_id] = thread_label

In [17]:
import random
print(all_thread_id[0])
random.shuffle(all_thread_id)
print(all_thread_id[0])

552784600502915072
553197181151481857


In [18]:
train_thread_id = all_thread_id[:len(all_thread_id) * 8 //10]
dev_thread_id = all_thread_id[len(all_thread_id) * 8 // 10:len(all_thread_id) * 9 //10]
test_thread_id = all_thread_id[len(all_thread_id) * 9 //10:]

In [20]:
def flattenStructure(structure: dict):
    '''
    展开字典形式存储的传播树结构, 获取所有节点的id
    Input:
        - structure: dict
    Output:
        - Ids: list
    '''
    Ids = []
    if not structure:
        return Ids # 当前dict为空，直接返回空列表
    Ids += list(structure.keys())
    for id in structure:
        if structure[id]: # 子dict不为空，递归地展开
            Ids += flattenStructure(structure[id])
    return Ids

In [24]:
train_post_id = []
train_thread_label = []
train_structure = {}
train_post = {}
for thread_id in train_thread_id:
    train_thread_label.append(id2label[thread_id])
    train_structure[thread_id] = all_structure[thread_id]

    train_post_id += flattenStructure(all_structure[thread_id])

for post_id in train_post_id:
    train_post[post_id] = all_post[post_id]

print(len(train_post_id))

with open(os.path.join('.', 'train', 'post_id.txt'), 'w') as f:
    for post_id in train_post_id:
        f.write(post_id)
        f.write('\n')

with open(os.path.join('.', 'train', 'posts.json'), 'w') as f:
    f.write(json.dumps(train_post))

with open(os.path.join('.', 'train', 'structures.json'), 'w') as f:
    f.write(json.dumps(train_structure))

with open(os.path.join('.', 'train', 'thread_id.txt'), 'w') as f:
    for thread_id in train_thread_id:
        f.write(thread_id)
        f.write('\n')

with open(os.path.join('.', 'train', 'thread_label.txt'), 'w') as f:
    for thread_label in train_thread_label:
        f.write(thread_label)
        f.write('\n')

81340


In [26]:
dev_post_id = []
dev_thread_label = []
dev_structure = {}
dev_post = {}
for thread_id in dev_thread_id:
    dev_thread_label.append(id2label[thread_id])
    dev_structure[thread_id] = all_structure[thread_id]

    dev_post_id += flattenStructure(all_structure[thread_id])

for post_id in dev_post_id:
    dev_post[post_id] = all_post[post_id]

print(len(dev_post_id))

with open(os.path.join('.', 'dev', 'post_id.txt'), 'w') as f:
    for post_id in dev_post_id:
        f.write(post_id)
        f.write('\n')

with open(os.path.join('.', 'dev', 'posts.json'), 'w') as f:
    f.write(json.dumps(dev_post))

with open(os.path.join('.', 'dev', 'structures.json'), 'w') as f:
    f.write(json.dumps(dev_structure))

with open(os.path.join('.', 'dev', 'thread_id.txt'), 'w') as f:
    for thread_id in dev_thread_id:
        f.write(thread_id)
        f.write('\n')

with open(os.path.join('.', 'dev', 'thread_label.txt'), 'w') as f:
    for thread_label in dev_thread_label:
        f.write(thread_label)
        f.write('\n')

10014


In [27]:
test_post_id = []
test_thread_label = []
test_structure = {}
test_post = {}
for thread_id in test_thread_id:
    test_thread_label.append(id2label[thread_id])
    test_structure[thread_id] = all_structure[thread_id]

    test_post_id += flattenStructure(all_structure[thread_id])

for post_id in test_post_id:
    test_post[post_id] = all_post[post_id]

print(len(test_post_id))

with open(os.path.join('.', 'test', 'post_id.txt'), 'w') as f:
    for post_id in test_post_id:
        f.write(post_id)
        f.write('\n')

with open(os.path.join('.', 'test', 'posts.json'), 'w') as f:
    f.write(json.dumps(test_post))

with open(os.path.join('.', 'test', 'structures.json'), 'w') as f:
    f.write(json.dumps(test_structure))

with open(os.path.join('.', 'test', 'thread_id.txt'), 'w') as f:
    for thread_id in test_thread_id:
        f.write(thread_id)
        f.write('\n')

with open(os.path.join('.', 'test', 'thread_label.txt'), 'w') as f:
    for thread_label in test_thread_label:
        f.write(thread_label)
        f.write('\n')

10392
